In [1]:
# | default_exp parser.extractors


# Parser Extractors
> Functions for extracting headers, links, and images from markdown and HTML content.

In [2]:
#| export
import re
from urllib.parse import urlparse
from seo_rat.parser.utils import is_special_url

IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".gif", ".bmp", ".svg", ".webp")


## Content Extraction

In [3]:
# | export
def extract_headers(content: str) -> list[dict]:
    "Extract all headers from markdown content."
    headings = []
    for line_number, line in enumerate(content.splitlines(), start=1):
        line = line.strip()
        for level in range(1, 7):
            prefix = "#" * level + " "
            if line.startswith(prefix):
                text = line.strip("#").strip()
                headings.append({"type": f"h{level}", "line_number": line_number,
                                 "content": text, "length": len(text)})
                break
    return headings


In [10]:
# | test
from fastcore.test import test_eq
from pathlib import Path

sample_dir = Path("sample")
if not sample_dir.exists():
    sample_dir = Path("../sample")

with open(sample_dir / "example.md") as f:
    content = f.read()
headers = extract_headers(content)
test_eq(len([h for h in headers if h["type"] == "h1"]), 2)
test_eq(headers[0]["content"], "This is me Kareem")


In [11]:
# | export
def extract_links(content: str) -> dict[str, dict]:
    """Extract all links with metadata"""
    links = {}
    lines = content.split("\n")
    for line_number, line in enumerate(lines, start=1):
        for match in re.finditer(r"(?<!!)\[(.*?)\]\((.*?)\)", line):
            title, url = match.groups()
            url = re.sub(r"""\s+["'].*?["']\s*$""", '', url).strip()
            if url not in links:
                links[url] = {"titles": [], "lines": []}
            links[url]["titles"].append(title)
            links[url]["lines"].append(line_number)

    # Also extract HTML links
    try:
        from bs4 import BeautifulSoup
    except Exception:
        return links

    soup = BeautifulSoup(content, "html.parser")
    for a in soup.find_all("a", href=True):
        url = a.get("href", "").strip()
        if not url:
            continue
        title = a.get_text(strip=True)
        if url not in links:
            links[url] = {"titles": [], "lines": []}
        links[url]["titles"].append(title)
        links[url]["lines"].append(-1)  # HTML line unknown
    return links


In [12]:
# | test
from fastcore.test import test_eq
from pathlib import Path

sample_dir = Path("sample")
if not sample_dir.exists():
    sample_dir = Path("../sample")
with open(sample_dir / "example.md") as file:
    content = file.read()
links = extract_links(content)
test_eq("https://emdadelgaz.com" in links, True)
test_eq("https://awazly.com/" in links, True)


In [13]:
# | export
def extract_images(content: str  # Markdown or HTML content
                   ) -> list[dict]:
    "Extract images with alt text from markdown and HTML."
    images = [{"alt_text": alt, "url": url} for alt, url in re.findall(r"\!\[(.*?)\]\((.*?)\)", content)]
    try:
        from bs4 import BeautifulSoup
        for img in BeautifulSoup(content, "html.parser").find_all("img"):
            if src := img.get("src", "").strip():
                images.append({"alt_text": img.get("alt", "").strip(), "url": src})
    except Exception:
        pass
    return images


In [14]:
# | export
def imgs_missing_alts(images: list[dict]  # List of image dicts from `extract_images`
                      ) -> list[str]:
    "Return URLs of images missing alt text."
    return [img["url"] for img in images if not img.get("alt_text")]


In [15]:
# | test
from fastcore.test import test_eq
from pathlib import Path

sample_dir = Path("sample")
if not sample_dir.exists():
    sample_dir = Path("../sample")
with open(sample_dir / "example.md") as file:
    content = file.read()
images = extract_images(content)
test_eq(len(images), 1)
test_eq(images[0]["alt_text"], "Iron man photo")


## Link Filtering

Utilities for classifying and filtering internal and external links.

In [16]:
# | export
def filter_internal_links(urls: list[str],  # List of URLs to filter
                          domain: str  # Domain to match against (e.g. 'example.com')
                          ) -> list[str]:
    "Filter for internal links, excluding images and special URLs."
    return [u for u in urls if not is_special_url(u)
            and not u.lower().endswith(IMAGE_EXTS)
            and (not u.startswith("http") or urlparse(u).netloc == domain)]


In [17]:
# | export
def filter_external_links(urls: list[str],  # List of URLs to filter
                          domain: str  # Site domain to exclude
                          ) -> list[str]:
    "Filter for external links only, excluding images and special URLs."
    internal = filter_internal_links(urls, domain)
    return [u for u in urls if u not in internal
            and not u.lower().endswith(IMAGE_EXTS)
            and not is_special_url(u)]